# task 2

their baseline for now. split by split, --split all skips april (87k zeros last time)

no labels anywhere, only LB tells. internet on

out: queue.csv

In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
REPO = Path("/tmp/trafficflowbench-public")
OUT = Path("/kaggle/working")


def run(*args):
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
    return r.stdout


if not REPO.exists():
    run("git", "clone", "https://github.com/jacky850/trafficflowbench-public.git", REPO)
print("organizers' repo commit:", run("git", "-C", REPO, "rev-parse", "HEAD").strip())


def stack(parts, dest):
    with open(dest, "w") as out:
        for i, p in enumerate(parts):
            with open(p) as f:
                header = f.readline()
                if i == 0:
                    out.write(header)
                out.writelines(f)


def count_rows(path):
    with open(path) as f:
        return sum(1 for _ in f) - 1

In [ ]:
EXPECTED_ROWS = 87_000

parts = []
for split in ["validation", "private"]:
    dest = Path(f"/tmp/queue_{split}.csv")
    run("python", REPO / "src/task2/build_task2_persistence_submission.py",
        "--release-root", REL, "--split", split, "--output", dest)
    q = pd.read_csv(dest, dtype=str)
    assert len(q) == EXPECTED_ROWS, (split, len(q))
    assert q["window_id"].str.contains(f"_{split}_").all(), f"{split}: windows from another split"
    print(split, q["queue_pred"].value_counts().to_dict())
    parts.append(dest)

stack(parts, OUT / "queue.csv")
print("queue.csv rows:", count_rows(OUT / "queue.csv"))